# 00 — Fashion-MNIST MLP：TTLinear 一層置換と Logits 等価性

配置想定：

```text
notebooks/
└── 30_tt_mps/
    ├── 00_fundamentals/
    │   └── ... Notebook 15 まで
    └── 10_fashion_mnist_mlp/
        └── 00_mlp_ttlinear_logits_equivalence.ipynb
```

## 今回の目的

Notebook 15 では、1つの dense `nn.Linear` と、truncation なし TT-matrix による direct contraction が、

$$
Y_{\mathrm{TT}}
\approx
Y_{\mathrm{dense}}
$$

となることを学びました。

今回はそれを **MLP の中へ一層だけ組み込んだとき、モデル全体の logits まで一致する理由と実装**を扱います。

中心となる等価性は、

$$
\boxed{
L_{\mathrm{TT}}(X)=L_{\mathrm{dense}}(X)
\quad\Longrightarrow\quad
f_{\mathrm{TT}}(X)=f_{\mathrm{dense}}(X)
}
$$

です。

ここで $L$ は置換対象の Linear、$f$ は MLP 全体を表します。

## 今回やらないこと

- TT-rank truncation
- 圧縮率の評価
- test accuracy の変化
- fine-tuning
- backward / optimizer
- TTLinear direct contraction の再実装
- TT-SVD の再実装


## 1. 前提：Notebook 15 から引き継ぐこと

PyTorch の `nn.Linear` は、

$$
X\in\mathbb R^{B\times n},
\qquad
W\in\mathbb R^{m\times n},
\qquad
b\in\mathbb R^m
$$

に対し、

$$
Y=XW^{\mathsf T}+b
$$

を計算します。

既存 TTLinear が表す weight を $W_{\mathrm{TT}}$ とすると、

$$
Y_{\mathrm{TT}}
=
XW_{\mathrm{TT}}^{\mathsf T}+b.
$$

truncation なし TT-SVD により、

$$
W_{\mathrm{TT}}\approx W
$$

なら、

$$
Y_{\mathrm{TT}}
\approx
Y_{\mathrm{dense}}.
$$

この Notebook では、この単層等価性を再実装せず、既存実装を使います。


## 2. 単層等価性から MLP 全体の等価性へ

第1層：

$$
Z_1=XW_1^{\mathsf T}+b_1
$$

activation：

$$
H=\phi(Z_1)
$$

第2層：

$$
L=HW_2^{\mathsf T}+b_2
$$

とします。$L$ が logits です。

shape は、

$$
X\in\mathbb R^{B\times n},
\quad
W_1\in\mathbb R^{h\times n},
\quad
W_2\in\mathbb R^{c\times h}.
$$

Fashion-MNIST では、

$$
c=10
$$

です。


## 3. なぜ一層が同じなら logits まで同じなのか

第1 dense Linear を truncation なし TTLinear で置換し、

$$
W_{1,\mathrm{TT}}=W_1
$$

かつ bias も同じなら、

$$
Z_{1,\mathrm{TT}}
=
XW_{1,\mathrm{TT}}^{\mathsf T}+b_1
=
XW_1^{\mathsf T}+b_1
=
Z_{1,\mathrm{dense}}.
$$

同じ deterministic activation $\phi$ なら、

$$
H_{\mathrm{TT}}
=
\phi(Z_{1,\mathrm{TT}})
=
\phi(Z_{1,\mathrm{dense}})
=
H_{\mathrm{dense}}.
$$

後段の $W_2,b_2$ が同じなら、

$$
L_{\mathrm{TT}}
=
H_{\mathrm{TT}}W_2^{\mathsf T}+b_2
=
H_{\mathrm{dense}}W_2^{\mathsf T}+b_2
=
L_{\mathrm{dense}}.
$$

したがって、

$$
\boxed{
Z_{1,\mathrm{TT}}\approx Z_{1,\mathrm{dense}}
\Rightarrow
H_{\mathrm{TT}}\approx H_{\mathrm{dense}}
\Rightarrow
L_{\mathrm{TT}}\approx L_{\mathrm{dense}}
}
$$

を段階ごとに検証します。


## 4. 安全な置換の考え方

元の dense model は比較基準として残します。

$$
\text{tt\_model}
=
\operatorname{deepcopy}(\text{dense\_model})
$$

とし、copy 側の対象 `nn.Linear` 一層だけを TTLinear に置換します。

これにより、

- dense reference model を保持
- 置換以外の差を混入させない
- 同じ入力 batch で比較

できます。


## 5. `eval()` / `no_grad()` / float64

今回の目的は forward correctness です。

比較条件を固定するため、

- CPU
- `torch.float64`
- `model.eval()`
- `torch.no_grad()`

を使います。

Dropout や BatchNorm がモデルに含まれる場合も、train/eval の差を持ち込まないようにします。


## 6. 実行環境


In [77]:
import copy
from math import prod

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
torch.set_default_dtype(torch.float64)

device = torch.device("cpu")
dtype = torch.float64

print("PyTorch:", torch.__version__)
print("default dtype:", torch.get_default_dtype())
print("device:", device)


PyTorch: 2.11.0+cu128
default dtype: torch.float64
device: cpu


## 7. 演習1 — 学習済み MLP の `nn.Linear` を列挙する

まず TT 化せず、モデル内のすべての `nn.Linear` について、

- module path
- `weight.shape`
- bias の有無
- `in_features`
- `out_features`
- parameter 数

を確認します。

Linear 層の parameter 数は、

$$
P
=
mn+
\begin{cases}
m & \text{biasあり}\\
0 & \text{biasなし}
\end{cases}
$$

です。


In [78]:
# TODO 1:
# リポジトリにある学習済み Fashion-MNIST MLP を
# CPU + float64 で読み込んでください。

from pathlib import Path

from nn_compression.models import MNISTMLP
from nn_compression.utils import find_project_root
#
# --- 設定 ---
# 直前セルの device / dtype を使う
project_root = find_project_root(Path.cwd())
checkpoint_path = (
    project_root
    / "models"
    / "10_svd"
    / "20_fashion_mnist_mlp"
    / "00_mlp_baseline_fixed_50epochs"
    / "00_mlp_baseline_fixed_50epochs.pt"
)
print("checkpoint_path:", checkpoint_path)
print("exists:", checkpoint_path.exists())

dense_model = MNISTMLP().to(device)
dense_model = dense_model.to(device=device, dtype=dtype)
dense_model.load_state_dict(torch.load(checkpoint_path, map_location=device))
dense_model.eval()
#
# dense_model.named_modules() を使って nn.Linear を列挙し、
# 次を表示してください。
# - module path
# - weight.shape
# - bias の有無
# - in_features
# - out_features
# - parameter 数
#
# まだ TT 化しないでください。
for name, module in dense_model.named_modules():
    if not isinstance(module, nn.Linear):
        continue
    n_params = module.weight.numel() + (
        module.bias.numel() if module.bias is not None else 0
    )
    print(
        f"path={name!r}, weight.shape={tuple(module.weight.shape)}, "
        f"bias={module.bias is not None}, "
        f"in={module.in_features}, out={module.out_features}, "
        f"n_params={n_params}"
    )

checkpoint_path: D:\dev\nn-compression-svd-dmrg\models\10_svd\20_fashion_mnist_mlp\00_mlp_baseline_fixed_50epochs\00_mlp_baseline_fixed_50epochs.pt
exists: True
path='fc1', weight.shape=(512, 784), bias=True, in=784, out=512, n_params=401920
path='fc2', weight.shape=(256, 512), bias=True, in=512, out=256, n_params=131328
path='fc3', weight.shape=(10, 256), bias=True, in=256, out=10, n_params=2570


## 8. 演習2 — 置換対象 hidden Linear を決める

最終 classifier ではなく、hidden `nn.Linear` の中から parameter 数が大きい層を選びます。

例えば対象層が、

$$
W\in\mathbb R^{512\times784}
$$

なら、

$$
m_{\mathrm{modes}}=(8,8,8,1),
$$

$$
n_{\mathrm{modes}}=(7,7,4,4)
$$

という分解が使えます。

ただし実際の layer shape が違う場合は、この値を流用してはいけません。

必ず、

$$
\prod_km_k=\text{out\_features},
\qquad
\prod_kn_k=\text{in\_features}
$$

を確認します。


In [79]:
from __future__ import annotations

import math
from itertools import product
from typing import Iterator


def ordered_factorizations(
    value: int,
    num_factors: int,
    min_factor: int = 2,
) -> list[tuple[int, ...]]:
    """value を num_factors 個の 2 以上の整数の順序付き積へ分解して返す。

    例:
        value=12, num_factors=2
        → (2, 6), (3, 4), (4, 3), (6, 2), ...
        （順序違いを別候補として残す）
    """
    if value < 1:
        raise ValueError(f"value must be positive, but got {value}.")
    if num_factors < 1:
        raise ValueError(
            f"num_factors must be at least 1, but got {num_factors}."
        )
    if min_factor < 1:
        raise ValueError(
            f"min_factor must be at least 1, but got {min_factor}."
        )

    def divisors_at_least(
        remaining: int,
        lower_bound: int,
    ) -> Iterator[int]:
        """remaining の約数のうち lower_bound 以上のものを昇順で yield する。"""
        for divisor in range(lower_bound, remaining + 1):
            if remaining % divisor == 0:
                yield divisor

    def recurse(
        remaining: int,
        factors_left: int,
        prefix: tuple[int, ...],
    ) -> list[tuple[int, ...]]:
        """残り積 remaining を factors_left 個へ分解し、prefix に続けて返す。"""
        # 最後の1因子は残りそのもの（条件を満たせば採用）
        if factors_left == 1:
            if remaining >= min_factor:
                return [prefix + (remaining,)]
            return []

        results: list[tuple[int, ...]] = []

        for factor in divisors_at_least(remaining, min_factor):
            next_remaining = remaining // factor

            # 残り因子を全部 min_factor 以上にできないなら枝刈り
            if next_remaining < min_factor ** (factors_left - 1):
                continue

            results.extend(
                recurse(
                    remaining=next_remaining,
                    factors_left=factors_left - 1,
                    prefix=prefix + (factor,),
                )
            )

        return results

    return recurse(
        remaining=value,
        factors_left=num_factors,
        prefix=(),
    )


def tt_physical_sizes(
    m_modes: tuple[int, ...],
    n_modes: tuple[int, ...],
) -> tuple[int, ...]:
    """TT-matrix の各 site における physical size q_k = m_k * n_k を返す。"""
    if len(m_modes) != len(n_modes):
        raise ValueError(
            "m_modes and n_modes must have the same number of modes."
        )

    return tuple(m_k * n_k for m_k, n_k in zip(m_modes, n_modes))


def shape_imbalance_score(
    m_modes: tuple[int, ...],
    n_modes: tuple[int, ...],
) -> float:
    """physical sizes の max/min を返す。1 に近いほど site 間が balanced。"""
    physical_sizes = tt_physical_sizes(m_modes, n_modes)

    if min(physical_sizes) <= 0:
        raise ValueError("All physical sizes must be positive.")

    return max(physical_sizes) / min(physical_sizes)


def tt_parameter_count(
    m_modes: tuple[int, ...],
    n_modes: tuple[int, ...],
    tt_ranks: tuple[int, ...],
) -> int:
    """TT-matrix core の総 parameter 数を計算する。

    各 core G^(k) の要素数は
        r_{k-1} * m_k * n_k * r_k
    """
    if len(m_modes) != len(n_modes):
        raise ValueError(
            "m_modes and n_modes must have the same number of modes."
        )

    num_cores = len(m_modes)

    if len(tt_ranks) != num_cores + 1:
        raise ValueError(
            "tt_ranks must have length len(m_modes) + 1. "
            f"Got len(tt_ranks)={len(tt_ranks)}, "
            f"len(m_modes)={num_cores}."
        )

    # 境界 rank は TT 形式どおり 1
    if tt_ranks[0] != 1 or tt_ranks[-1] != 1:
        raise ValueError(
            f"TT boundary ranks must be 1, but got {tt_ranks}."
        )

    return sum(
        left_rank * m_k * n_k * right_rank
        for left_rank, m_k, n_k, right_rank in zip(
            tt_ranks[:-1],
            m_modes,
            n_modes,
            tt_ranks[1:],
        )
    )


# ------------------------------------------------------------
# Fashion-MNIST MLP の第1層:
# nn.Linear(784, 512)
# weight.shape = (512, 784)
#
# ここでは「どんな (m_modes, n_modes) があり得るか」を列挙し、
# site 間のバランスと TT パラメータ数で並べる。
# ------------------------------------------------------------
output_dim = 512
input_dim = 784
num_cores = 3

# out_features / in_features をそれぞれ num_cores 個の因子へ分解
m_factorizations = ordered_factorizations(
    value=output_dim,
    num_factors=num_cores,
)

n_factorizations = ordered_factorizations(
    value=input_dim,
    num_factors=num_cores,
)

# 例として内部 TT-rank を固定する。
# r0 = r3 = 1, r1 = r2 = 8
tt_ranks = (1, 8, 8, 1)

# (imbalance, param_count, m_modes, n_modes, physical_sizes)
candidates: list[
    tuple[
        float,
        int,
        tuple[int, ...],
        tuple[int, ...],
        tuple[int, ...],
    ]
] = []

# すべての (m_modes, n_modes) 組み合わせを評価
for m_modes, n_modes in product(m_factorizations, n_factorizations):
    physical_sizes = tt_physical_sizes(m_modes, n_modes)
    imbalance = shape_imbalance_score(m_modes, n_modes)

    parameter_count = tt_parameter_count(
        m_modes=m_modes,
        n_modes=n_modes,
        tt_ranks=tt_ranks,
    )

    candidates.append(
        (
            imbalance,
            parameter_count,
            m_modes,
            n_modes,
            physical_sizes,
        )
    )

# まず imbalance が小さい順、
# 同点なら TT parameter 数が少ない順に並べる。
candidates.sort(key=lambda item: (item[0], item[1]))

# dense Linear の parameter 数（W + bias）と比較用
dense_parameter_count = output_dim * input_dim + output_dim

print(f"dense weight shape      : ({output_dim}, {input_dim})")
print(f"dense Linear parameters : {dense_parameter_count:,}")
print(f"fixed TT ranks          : {tt_ranks}")
print()

# 上位候補だけ表示
for index, (
    imbalance,
    parameter_count,
    m_modes,
    n_modes,
    physical_sizes,
) in enumerate(candidates[:10], start=1):
    compression_ratio = dense_parameter_count / (parameter_count + output_dim)

    print(f"candidate {index}")
    print(f"  m_modes          = {m_modes}")
    print(f"  n_modes          = {n_modes}")
    print(f"  physical_sizes   = {physical_sizes}")
    print(f"  imbalance        = {imbalance:.6f}")
    print(f"  TT params + bias = {parameter_count + output_dim:,}")
    print(f"  compression ratio= {compression_ratio:.6f}")
    print()


dense weight shape      : (512, 784)
dense Linear parameters : 401,920
fixed TT ranks          : (1, 8, 8, 1)

candidate 1
  m_modes          = (2, 8, 32)
  n_modes          = (49, 8, 2)
  physical_sizes   = (98, 64, 64)
  imbalance        = 1.531250
  TT params + bias = 5,904
  compression ratio= 68.075881

candidate 2
  m_modes          = (2, 16, 16)
  n_modes          = (49, 4, 4)
  physical_sizes   = (98, 64, 64)
  imbalance        = 1.531250
  TT params + bias = 5,904
  compression ratio= 68.075881

candidate 3
  m_modes          = (2, 32, 8)
  n_modes          = (49, 2, 8)
  physical_sizes   = (98, 64, 64)
  imbalance        = 1.531250
  TT params + bias = 5,904
  compression ratio= 68.075881

candidate 4
  m_modes          = (8, 32, 2)
  n_modes          = (8, 2, 49)
  physical_sizes   = (64, 64, 98)
  imbalance        = 1.531250
  TT params + bias = 5,904
  compression ratio= 68.075881

candidate 5
  m_modes          = (16, 16, 2)
  n_modes          = (4, 4, 49)
  physical_size

In [80]:
from __future__ import annotations

import math


def divisors(value: int, min_divisor: int) -> list[int]:
    """value の約数のうち min_divisor 以上のものを昇順で返す。"""
    if value < 1:
        raise ValueError(f"value must be positive, but got {value}.")
    if min_divisor < 1:
        raise ValueError(
            f"min_divisor must be at least 1, but got {min_divisor}."
        )

    result: list[int] = []

    # sqrt(value) まで見ればペア約数で全部拾える
    for divisor in range(min_divisor, math.isqrt(value) + 1):
        if value % divisor == 0:
            result.append(divisor)

            paired_divisor = value // divisor
            if paired_divisor != divisor:
                result.append(paired_divisor)

    # value 自身も候補に含める（例: 最後の因子）
    if value >= min_divisor:
        result.append(value)

    return sorted(set(result))


def unordered_factorizations(
    value: int,
    num_modes: int,
    min_factor: int = 2,
) -> list[tuple[int, ...]]:
    """value を num_modes 個の非減少な整数因子の積へ分解して返す。

    非減少（a <= b <= c ...）にすることで、
    (2, 6) と (6, 2) のような順序違いを同一視する。
    """
    if value < 1:
        raise ValueError(f"value must be positive, but got {value}.")
    if num_modes < 1:
        raise ValueError(
            f"num_modes must be at least 1, but got {num_modes}."
        )
    if min_factor < 1:
        raise ValueError(
            f"min_factor must be at least 1, but got {min_factor}."
        )

    def recurse(
        remaining: int,
        factors_left: int,
        lower_bound: int,
    ) -> list[tuple[int, ...]]:
        """非減少制約つきで factor tuple を再帰列挙する。

        lower_bound:
            直前までに選んだ因子。これ以上の値だけを次に選ぶ。
        """
        # 最後の1因子は残りそのもの
        if factors_left == 1:
            if remaining >= lower_bound:
                return [(remaining,)]
            return []

        candidates: list[tuple[int, ...]] = []

        for factor in divisors(remaining, lower_bound):
            if factor > remaining:
                continue

            next_remaining = remaining // factor

            # 残り因子をすべて「今の factor 以上」にできないなら枝刈り
            if next_remaining < factor ** (factors_left - 1):
                continue

            for suffix in recurse(
                remaining=next_remaining,
                factors_left=factors_left - 1,
                lower_bound=factor,  # 非減少: 次は factor 以上
            ):
                candidates.append((factor,) + suffix)

        return candidates

    return recurse(
        remaining=value,
        factors_left=num_modes,
        lower_bound=min_factor,
    )


def factor_balance_score(
    factors: tuple[int, ...],
) -> tuple[float, int, int]:
    """小さいほど均等な factorization になる score を返す。

    比較キー（辞書式）:
        1. max/min が小さいほど均等
        2. 同率なら max-min が小さいほど均等
        3. 同率なら最大 factor が小さい方を優先
    """
    if len(factors) == 0:
        raise ValueError("factors must not be empty.")
    if min(factors) <= 0:
        raise ValueError(f"all factors must be positive, but got {factors}.")

    minimum = min(factors)
    maximum = max(factors)

    return (
        maximum / minimum,
        maximum - minimum,
        maximum,
    )


def balanced_modes(
    value: int,
    num_modes: int,
    min_factor: int = 2,
) -> tuple[int, ...]:
    """value を num_modes 個のなるべく均等な整数因子へ自動分解する。

    例:
        balanced_modes(512, 3) → (8, 8, 8)
        balanced_modes(784, 3) → なるべく均等な3因子
    """
    candidates = unordered_factorizations(
        value=value,
        num_modes=num_modes,
        min_factor=min_factor,
    )

    if len(candidates) == 0:
        raise ValueError(
            f"Cannot factorize value={value} into {num_modes} factors "
            f"that are each at least {min_factor}."
        )

    # score が最小の分解を採用
    best = min(candidates, key=factor_balance_score)

    if math.prod(best) != value:
        raise RuntimeError(
            f"Internal error: product of {best} is not {value}."
        )

    return best


In [81]:
# TODO 2:
# TODO 1 の一覧から対象 hidden nn.Linear を1つ決めてください。
# --- 設定 ---
# 直前セルの dense_model を使う
# hidden: fc1 / fc2（classifier の fc3 は対象外）
from math import prod
#
target_path = "fc1"
target_linear = dict(dense_model.named_modules())[target_path]

# target_linear が nn.Linear であることを確認してください。
assert isinstance(target_linear, nn.Linear),(
    "not Linear"
)
#
num_tt_cores = 4
# 対象層に合う m_modes / n_modes を決め、
m ,n = target_linear.weight.shape
m_modes = balanced_modes(m,num_tt_cores)
n_modes = balanced_modes(n,num_tt_cores)

assert prod(m_modes) == target_linear.out_features
assert prod(n_modes) == target_linear.in_features



## 9. 演習3 — 学習済み dense Linear から truncation なし TTLinear を作る

ここでは Notebook 15 までに実装済みの、

- dense → TT-matrix
- truncation なし TT-SVD
- TTLinear

を再利用します。

条件は、

- truncation なし
- $\varepsilon=0$
- bias は dense layer と同じ値
- CPU
- float64

です。

TT-SVD や direct contraction は再実装しません。


In [82]:
# TODO 3:
# 既存のプロジェクト実装を使って、
# target_linear.weight / target_linear.bias から
# truncation なし TTLinear を作ってください。
#
# --- import ---
# 注: src に TTLinear (nn.Module) はまだない。
# truncation なし変換 + forward は次の functional API を使う。
from nn_compression.compression import (
    dense_to_tt_matrix_cores,  # W → TT-matrix cores（max_rank=None で truncation なし）
    tt_linear_forward,         # cores で direct contraction
    tt_matrix_to_dense,        # 検証用: cores → dense W
    tt_matrix_num_parameters,
    tt_matrix_ranks,
)
tt_cores = dense_to_tt_matrix_cores(
    target_linear.weight.detach(),
    m_modes,
    n_modes,
    max_rank=None,
)

# src に TTLinear Module がないので、薄い wrapper をノート内で定義する
class TTLinear(nn.Module):
    """TT-matrix cores + bias で nn.Linear 相当の forward を行う層。"""

    def __init__(
        self,
        cores: list[torch.Tensor],
        out_modes: tuple[int, ...] | list[int],
        in_modes: tuple[int, ...] | list[int],
        bias: torch.Tensor | None,
    ):
        super().__init__()
        # Parameter ではなく buffer: 今は学習せず forward 等価性だけ見る
        self.cores = nn.ParameterList(
            [nn.Parameter(c.clone(), requires_grad=False) for c in cores]
        )
        self.out_modes = tuple(int(m) for m in out_modes)
        self.in_modes = tuple(int(n) for n in in_modes)
        if bias is None:
            self.bias = None
        else:
            self.register_buffer("bias", bias.detach().clone())

    @property
    def in_features(self) -> int:
        return int(prod(self.in_modes))

    @property
    def out_features(self) -> int:
        return int(prod(self.out_modes))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return tt_linear_forward(
            x,
            list(self.cores),
            self.out_modes,
            self.in_modes,
            bias=self.bias,
        )


tt_layer = TTLinear(
    tt_cores,
    m_modes,
    n_modes,
    bias=None if target_linear.bias is None else target_linear.bias,
).to(device=device, dtype=dtype)
tt_layer.eval()

print("tt ranks:", tt_matrix_ranks(list(tt_layer.cores)))
print("tt params:", tt_matrix_num_parameters(list(tt_layer.cores)))
print("in/out:", tt_layer.in_features, tt_layer.out_features)


tt ranks: (1, 16, 256, 56, 1)
tt params: 470336
in/out: 784 512


## 10. 演習4 — 学習済み対象層「単体」で等価性を確認する

Notebook 15 は小さな random 例でした。

ここでは実際の学習済み weight に対して、

$$
Y_{\mathrm{dense}}
=
\operatorname{F.linear}(X_{\mathrm{layer}},W,b)
$$

と、

$$
Y_{\mathrm{TT}}
=
\operatorname{TTLinear}(X_{\mathrm{layer}})
$$

を比較します。

ここが通れば、次のモデル全体検証で失敗した場合に、

> TTLinear 自体ではなく module replacement / forward 経路側を疑う

ことができます。


In [83]:
# TODO 4:
# 対象 Linear に入る shape と同じ input batch を用意し、
# dense / TT の単層出力を比較してください。
#
# --- 設定 ---
# 直前セルの tt_layer / target_linear / device / dtype を使う
B_layer = 4
x_layer = torch.randn(
    B_layer,
    target_linear.in_features,
    dtype=dtype,
    device=device,
)

y_tt_layer = tt_layer(x_layer)
y_dense_layer =target_linear(x_layer)
# 最終的に:
torch.testing.assert_close(
    y_tt_layer,
    y_dense_layer,
    rtol=1e-11,
    atol=1e-12,
)



## 11. Module path と置換位置

`named_modules()` の path が例えば、

```text
classifier.0
```

なら、

- 親 module：`classifier`
- 子 module：`0`

です。

通常 attribute の、

```text
fc1
```

の場合もあります。

`nn.Sequential` / `nn.ModuleList` / 通常 attribute のどれで保持されているかを確認し、copy 側だけを置換します。


## 12. 演習5 — `deepcopy` して一層だけ置換する

作るのは、

```text
dense_model
    └── 全層 dense

tt_model = deepcopy(dense_model)
    └── target_path だけ TTLinear
```

です。

確認すること：

- `dense_model` の対象層は元の `nn.Linear`
- `tt_model` の対象層だけ TTLinear
- 対象以外の module 構造は同じ
- 両方 `eval()`
- 両方 CPU / float64


In [84]:
# TODO 5:
# dense_model は比較基準として残し、
# コピー側 tt_model の対象層だけを TTLinear に差し替える。

# 1) 全パラメータ・構造を複製（dense_model 自体は触らない）
tt_model = copy.deepcopy(dense_model)

# ------------------------------------------------------------
# module path ユーティリティ
#   "fc1"           → model.fc1
#   "classifier.0"  → model.classifier[の子 0] のようなネストにも対応
# ------------------------------------------------------------
def get_module_by_path(model: nn.Module, path: str) -> nn.Module:
    """path で指定した子 module を取り出す。"""
    module: nn.Module = model
    for part in path.split("."):
        module = getattr(module, part)
    return module


def set_module_by_path(
    model: nn.Module,
    path: str,
    new_module: nn.Module,
    *,
    verbose: bool = False,
) -> None:
    """path で指定した子 module を new_module に置き換える。"""
    parts = path.split(".")
    if verbose:
        print("[set_module_by_path] path =", repr(path))
        print("[set_module_by_path] parts =", parts)
        print("[set_module_by_path] new_module =", type(new_module).__name__)

    parent: nn.Module = model
    if verbose:
        print("[set_module_by_path] start parent =", type(parent).__name__)

    # 最後の名前のひとつ手前までが親
    for part in parts[:-1]:
        parent = getattr(parent, part)
        if verbose:
            print(
                "[set_module_by_path] descend to",
                repr(part),
                "->",
                type(parent).__name__,
            )

    child_name = parts[-1]
    old_module = getattr(parent, child_name)
    if verbose:
        print(
            "[set_module_by_path] replace",
            f"{type(parent).__name__}.{child_name}:",
            type(old_module).__name__,
            "->",
            type(new_module).__name__,
        )
    setattr(parent, child_name, new_module)
    if verbose:
        print(
            "[set_module_by_path] done:",
            type(getattr(parent, child_name)).__name__,
        )


# 2) 本体: tt_model 側の target_path だけ tt_layer に置換
#    fc1 固定なら実質 tt_model.fc1 = tt_layer と同じ
set_module_by_path(tt_model, target_path, tt_layer, verbose=True)

# 3) forward 比較用に eval / dtype / device を揃える
dense_model.eval()
tt_model.eval()
dense_model = dense_model.to(device=device, dtype=dtype)
tt_model = tt_model.to(device=device, dtype=dtype)

# ------------------------------------------------------------
# 4) 置換結果の検証
# ------------------------------------------------------------
dense_target = get_module_by_path(dense_model, target_path)
tt_target = get_module_by_path(tt_model, target_path)

# dense 側の対象は元の nn.Linear のまま
assert isinstance(dense_target, nn.Linear), type(dense_target)
# tt 側の対象だけ TTLinear になっている
assert isinstance(tt_target, TTLinear), type(tt_target)
# 差し込んだオブジェクトが、作った tt_layer そのもの
assert tt_target is tt_layer

# 対象以外の module は型が一致すること
# （置換の巻き添えで relu などが変わっていないか）
for name, dense_module in dense_model.named_modules():
    if name == "" or name == target_path:
        continue
    tt_module = dict(tt_model.named_modules())[name]
    assert type(dense_module) is type(tt_module), (
        f"path={name!r}: dense={type(dense_module)}, tt={type(tt_module)}"
    )

# dense 側の対象 weight が、置換作業で書き換わっていないこと
torch.testing.assert_close(
    dense_target.weight,
    target_linear.weight,
    rtol=0.0,
    atol=0.0,
)

print("OK: dense_model unchanged; tt_model replaced only", repr(target_path))
print("dense target:", type(dense_target).__name__)
print("tt target:", type(tt_target).__name__)


[set_module_by_path] path = 'fc1'
[set_module_by_path] parts = ['fc1']
[set_module_by_path] new_module = TTLinear
[set_module_by_path] start parent = MNISTMLP
[set_module_by_path] replace MNISTMLP.fc1: Linear -> TTLinear
[set_module_by_path] done: TTLinear
OK: dense_model unchanged; tt_model replaced only 'fc1'
dense target: Linear
tt target: TTLinear


## 13. なぜ中間出力も比較するのか

最終 logits だけでなく、

1. 対象 Linear 直後
2. activation 直後
3. logits

を順に比較します。

理論上、

$$
Z_{\mathrm{TT}}\approx Z_{\mathrm{dense}}
$$

なら、

$$
\phi(Z_{\mathrm{TT}})
\approx
\phi(Z_{\mathrm{dense}})
$$

であり、その後の module が同じなら、

$$
L_{\mathrm{TT}}\approx L_{\mathrm{dense}}
$$

です。

この Notebook ではまだ Lipschitz 誤差評価には進みません。


## 14. 演習6 — 同一 input batch を用意する

dense / TT の両方に **同じ tensor** を渡します。

Fashion-MNIST MLP が model 内で flatten するなら `(B,1,28,28)`、外で flatten 済みなら、

$$
X\in\mathbb R^{B\times784}
$$

を使います。

correctness 検証だけなら dataset download は不要です。


In [85]:
# TODO 6:
# dense_model が期待する input shape に合わせて、
# 1つの float64 input batch を作ってください。
#
# x_batch = ...
#
# dense 用と TT 用に別々の random tensor を作らないでください。
#
# --- 設定 ---
# 直前セルの dense_model / device / dtype を使う
# MNISTMLP は (B, 1, 28, 28) を受け取り、forward 内で 784 に flatten する
B = 4
x_batch = torch.randn(B, 1, 28, 28, dtype=dtype, device=device)
print("x_batch.shape:", tuple(x_batch.shape))


x_batch.shape: (4, 1, 28, 28)


## 15. 演習7 — Linear → activation → logits を段階別に比較する

可能なら forward hook、またはモデル構造に合わせた明示的 forward により、

- 対象層直後
- activation 直後
- logits

を取得します。

各段階で、

```python
torch.testing.assert_close(
    actual,
    expected,
    rtol=1e-11,
    atol=1e-12,
)
```

を使います。

確認したい流れは、

$$
Z_{\mathrm{TT}}
\approx
Z_{\mathrm{dense}}
$$

$$
\Downarrow
$$

$$
H_{\mathrm{TT}}
\approx
H_{\mathrm{dense}}
$$

$$
\Downarrow
$$

$$
L_{\mathrm{TT}}
\approx
L_{\mathrm{dense}}.
$$


In [86]:
# TODO 7:
# dense_model / tt_model を model.eval() にし、
dense_model.eval()
tt_model.eval()
# torch.no_grad() の中で同じ x_batch を入力してください。
with torch.no_grad():
    x_flat = x_batch.view(x_batch.size(0), -1)
    # 1. target Linear 直後
    Z_dense = dense_model.fc1(x_flat)
    Z_tt = tt_model.fc1(x_flat)
    torch.testing.assert_close(
        Z_dense,
        Z_tt,
        rtol=1e-11,
        atol=1e-12,
    )
    print("max_abs_error (target Linear):", (Z_dense - Z_tt).abs().max().item())
    # 2. activation 直後
    H_dense = dense_model.relu1(Z_dense)
    H_tt = tt_model.relu1(Z_tt)
    torch.testing.assert_close(
        H_dense,
        H_tt,
        rtol=1e-11,
        atol=1e-12,
    )
    print("max_abs_error (activation):", (H_dense - H_tt).abs().max().item())
    # 3. logits
    logits_dense = dense_model.fc3(dense_model.relu2(dense_model.fc2(H_dense)))
    logits_tt = tt_model.fc3(tt_model.relu2(tt_model.fc2(H_tt)))
    torch.testing.assert_close(
        logits_dense,
        logits_tt,
        rtol=1e-11,
        atol=1e-12,
    )
    print("max_abs_error (logits):", (logits_dense - logits_tt).abs().max().item())
#

# 最低限、最終 logits は必ず比較してください。
#
# 各段階の max absolute error も表示してください。


max_abs_error (target Linear): 1.3855583347321954e-13
max_abs_error (activation): 9.725553695716371e-14
max_abs_error (logits): 1.0231815394945443e-12


## 16. 演習8 — prediction の一致を補助確認する

logits が一致すれば、

$$
\hat y
=
\operatorname*{argmax}_c L_c
$$

も通常一致します。

ただしここでは accuracy 評価はしません。

目的は、同じ batch に対して置換前後の分類結果も同じであることの補助確認です。


In [87]:
# TODO 8:
# logits_dense / logits_tt から prediction を計算し、
# 同じであることを確認してください。
#
pred_dense = logits_dense.argmax(dim=1)
pred_tt = logits_tt.argmax(dim=1)
assert torch.equal(pred_dense, pred_tt) ,(
    "pred_dense and pred_tt are mismatch"
)
# ここでは test accuracy は計算しません。


## 17. 失敗位置から原因を切り分ける

### 単層比較から失敗

候補：

- dense → TT 変換
- mode 順序
- truncation が無効になっていない
- TTLinear forward
- bias
- dtype

### 単層は通るが、モデル内の対象層出力が違う

候補：

- module path
- flatten / reshape
- forward 順序
- 置換対象の取り違え

### 対象層出力は通るが activation 後が違う

候補：

- train/eval mode
- Dropout
- BatchNorm
- inplace activation
- hook の取得位置

### activation まで通るが logits が違う

対象層より後ろに意図しない差が入っていないか確認します。


## 18. 演習9 — 「一層だけ置換した」ことを確認する

dense model と TT model の差分が、

$$
\boxed{
\text{target Linear}
\rightarrow
\text{TTLinear}
}
$$

だけであることを確認します。

特に、

- target path 以外の module 構造
- target より後段の parameter
- 元の dense model

が変更されていないことを検証してください。


In [89]:
dense_modules = dict(dense_model.named_modules())
tt_modules = dict(tt_model.named_modules())

for name, dense_module in dense_modules.items():
    if name == "":
        continue
    tt_module = tt_modules[name]

    if name == target_path:
        assert isinstance(dense_module, nn.Linear)
        assert isinstance(tt_module, TTLinear)
        continue

    assert type(dense_module) is type(tt_module)
    if isinstance(dense_module, nn.Linear):
        torch.testing.assert_close(dense_module.weight, tt_module.weight, rtol=0.0, atol=0.0)
        torch.testing.assert_close(dense_module.bias, tt_module.bias, rtol=0.0, atol=0.0)

# dense 側の対象層そのものが壊れていないこと
assert torch.equal(dense_model.fc1.weight, target_linear.weight)  # path が fc1 のとき


## 19. 今回の完了条件

次がすべて通れば完了です。

1. 対象 hidden Linear を特定
2. truncation なし TTLinear を生成
3. 学習済み対象層単体で一致
4. dense model を保持したまま copy 側だけ置換
5. 対象 Linear 出力が一致
6. activation 後が一致
7. logits が一致
8. prediction も一致

最重要条件は、

$$
\boxed{
L_{\mathrm{TT}}
\approx
L_{\mathrm{dense}}
}
$$

です。

ここまででは、まだ rank truncation による「圧縮」は行っていません。


## 20. 次段階への接続

次に rank truncation を導入すると、

$$
\widetilde W=W+\Delta W
$$

となり、

$$
\Delta Y
=
X(\Delta W)^{\mathsf T}
$$

という出力誤差が生じます。

その後に、

- hidden pre-activation error
- activation error
- logits error
- prediction
- test accuracy
- fine-tuning

へ進みます。
